# CV3 – Computer Vision Sprint 3  
**CNN-Based Image Classification + YOLO Object Detection & Instance Segmentation**

## Project Overview
This project focuses on building computer vision models from scratch and extending them to object detection and instance segmentation.  

The project is divided into two major parts:

**Part A – CNN Classification**  
Binary classification of images into:
- Seat Cover  
- Toilet  

Four CNN models are trained:
- Seat Cover (No Augmentation)  
- Seat Cover (With Augmentation)  
- Toilet (No Augmentation)  
- Toilet (With Augmentation)  

**Part B – YOLO Detection & Instance Segmentation**  
A YOLO-based model is fine-tuned locally on the Toilet dataset to perform object detection and instance segmentation.

## Objectives
- Design and train CNNs from scratch using PyTorch  
- Apply aspect-ratio preserved padding and resizing  
- Evaluate models using accuracy, precision, recall, F1-score, and confusion matrix  
- Analyze misclassified samples and explain errors  
- Learn labeling and annotation concepts and tools  
- Train YOLO locally without using Roboflow  
- Export best models and deploy via API and GUI  

## Constraints
- No transfer learning for CNN  
- Augmentation applied only to training set  
- Fixed dataset splits  
- Training performed locally  
- Must understand every line of code  

---



In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import random
import shutil
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image
from torchvision.datasets import ImageFolder
from torchsummary import summary
from torchvision.utils import save_image

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns

# Device check
device = torch.device("cuda" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

print("Using device:", device)


# ----------------------------
# Dataset Root
# ----------------------------
DATASET_ROOT = "data"   # change only if needed

print("Dataset root:", DATASET_ROOT)


In [ ]:
# ============================
# CV3 - LEVEL 1: PREPROCESSING
# ============================

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image

IMG_SIZE = 224
BATCH_SIZE = 32

# ----------------------------
# Padding + Resize Transform
# ----------------------------
class PadResize:
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        w, h = img.size
        max_side = max(w, h)

        pad_w = (max_side - w) // 2
        pad_h = (max_side - h) // 2

        padding = (pad_w, pad_h, max_side - w - pad_w, max_side - h - pad_h)
        img = transforms.functional.pad(img, padding, fill=0)
        img = img.resize((self.size, self.size))
        return img

# ----------------------------
# Transforms
# ----------------------------
train_transform_aug = transforms.Compose([
    PadResize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

train_transform_noaug = transforms.Compose([
    PadResize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

val_test_transform = transforms.Compose([
    PadResize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

print("Transforms created")


In [ ]:
# ----------------------------
# Dataset Paths
# ----------------------------
toilet_train_dir = "data/splits/Toilets/train"
toilet_val_dir   = "data/splits/Toilets/val"
toilet_test_dir  = "data/splits/Toilets/test"

# ----------------------------
# Toilet Datasets
# ----------------------------
toilet_train_noaug = datasets.ImageFolder(toilet_train_dir, transform=train_transform_noaug)
toilet_train_aug   = datasets.ImageFolder(toilet_train_dir, transform=train_transform_aug)
toilet_val = datasets.ImageFolder(toilet_val_dir, transform=val_test_transform)
toilet_test = datasets.ImageFolder(toilet_test_dir, transform=val_test_transform)

print("Datasets loaded")


In [ ]:
from torch.utils.data import DataLoader

toilet_train_loader_noaug = DataLoader(toilet_train_noaug, batch_size=32, shuffle=True)
toilet_train_loader_aug   = DataLoader(toilet_train_aug, batch_size=32, shuffle=True)
toilet_val_loader = DataLoader(toilet_val, batch_size=32, shuffle=False)
toilet_test_loader = DataLoader(toilet_test, batch_size=32, shuffle=False)


In [ ]:
images, labels = next(iter(toilet_train_loader_noaug))
print(images.shape)
print(labels.shape)

## LEVEL 2 — Improved CNN Architecture

This level defines an improved Convolutional Neural Network (CNN) architecture for image classification.

Compared to the previous Sprint-2 CNN, the architecture is enhanced by:

- Adding an extra convolutional block to increase depth  
- Using Batch Normalization after each convolution to stabilize training  
- Increasing feature channels up to 256  
- Using explicit fully connected layers instead of lazy layers  

These improvements allow the network to learn richer visual features and achieve better classification performance on Seat Cover and Toilet images.

The model is trained from scratch using PyTorch.


In [ ]:
class ImageClassifierCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


### ImprovedCNN_1 (Deeper + Regularized CNN)
- Adds extra convolution layers to learn richer features
- BatchNorm after convolutions for training stability
- Higher dropout to reduce overfitting
- Same input pipeline, architecture-only improvement vs baseline

In [ ]:
class ImprovedCNN_1(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3 (extra depth)
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.7),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

### ImprovedCNN_2 (Global Pooling CNN)
- Deeper channel progression (64 → 512) for stronger feature learning
- Uses AdaptiveAvgPool2d (global pooling) instead of huge flatten layers
- Fewer parameters in the classifier, better generalization
- More modern head design while keeping the same training setup

In [ ]:
class ImprovedCNN_2(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        return self.classifier(x)

In [ ]:
EPOCHS = 50
LR = 0.001
BATCH_SIZE = 32

criterion = nn.CrossEntropyLoss()

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

toilet_train_ds_aug = ImageFolder(
    root="data/splits/Toilets/train",
    transform=train_transform_aug
)

toilet_val_ds = ImageFolder(
    root="data/splits/Toilets/val",
    transform=val_test_transform
)

toilet_train_loader_aug = DataLoader(
    toilet_train_ds_aug, batch_size=BATCH_SIZE, shuffle=True
)

toilet_val_loader = DataLoader(
    toilet_val_ds, batch_size=BATCH_SIZE, shuffle=False
)

print("Toilet train batches:", len(toilet_train_loader_aug))

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, name):

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(EPOCHS):

        # TRAIN
        model.train()
        running_loss = 0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = outputs.argmax(1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct_train / total_train

        # VAL
        model.eval()
        running_val_loss = 0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item()
                preds = outputs.argmax(1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)

        val_loss = running_val_loss / len(val_loader)
        val_acc = correct_val / total_val

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(
            f"{name} | Epoch {epoch+1}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}"
        )

    return train_losses, val_losses, train_accs, val_accs

In [ ]:
baseline_model = ImageClassifierCNN(num_classes=3).to(device)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=LR)

imp1_model = ImprovedCNN_1(num_classes=3).to(device)
imp1_optimizer = torch.optim.Adam(imp1_model.parameters(), lr=LR)

imp2_model = ImprovedCNN_2(num_classes=3).to(device)
imp2_optimizer = torch.optim.Adam(imp2_model.parameters(), lr=LR)

In [ ]:
toilet_model_aug = ImageClassifierCNN(num_classes=3).to(device)
toilet_optimizer_aug = torch.optim.Adam(toilet_model_aug.parameters(), lr=LR)

# Baseline
base_losses, base_val_losses, base_accs, base_val_accs = train_model(
    baseline_model,
    toilet_train_loader_aug,
    toilet_val_loader,
    baseline_optimizer,
    "Baseline"
)

In [ ]:
model = ImprovedCNN_1(num_classes=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


# Improved 1
imp1_losses, imp1_val_losses, imp1_accs, imp1_val_accs = train_model(
    imp1_model,
    toilet_train_loader_aug,
    toilet_val_loader,
    imp1_optimizer,
    "ImprovedCNN_1"
)

In [ ]:
model = ImprovedCNN_2(num_classes=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Improved 2
imp2_losses, imp2_val_losses, imp2_accs, imp2_val_accs = train_model(
    imp2_model,
    toilet_train_loader_aug,
    toilet_val_loader,
    imp2_optimizer,
    "ImprovedCNN_2"
)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, EPOCHS+1)

plt.figure()
plt.plot(epochs, base_val_accs, marker="o", label="Baseline")
plt.plot(epochs, imp1_val_accs, marker="o", label="ImprovedCNN_1")
plt.plot(epochs, imp2_val_accs, marker="o", label="ImprovedCNN_2")

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Model Comparison")
plt.legend()
plt.show()

## Object Detection and Instance Segmentation

Object detection is a computer vision task that identifies and localizes objects within an image by drawing bounding boxes around each object and assigning a class label.

Instance segmentation is a more detailed task where, in addition to detecting objects, each individual object is segmented at the pixel level. This means the exact shape of each object is identified, not just its bounding box.

### How Object Detection Can Be Done
Object detection can be performed using deep learning models that learn to predict bounding boxes and class labels directly from images. Common approaches include:

- Two-stage detectors (e.g., R-CNN family), where region proposals are generated first and then classified.
- One-stage detectors (e.g., YOLO), where detection is performed in a single forward pass, making them faster.

### How Instance Segmentation Can Be Done
Instance segmentation models extend object detection by also predicting a pixel-level mask for each detected object. This is commonly achieved using:

- Mask-based architectures such as Mask R-CNN
- YOLO-based segmentation models that output both bounding boxes and segmentation masks

These approaches allow objects to be detected and precisely separated from the background.


## Labeling and Annotation

Labeling is the process of assigning a class name to an image or object.  
For example, assigning the label "toilet" to an image.

Annotation is the process of marking the exact location and shape of objects inside an image. This is required for object detection and instance segmentation.

Common annotation types:
- Bounding boxes (for object detection)
- Segmentation masks (for instance segmentation)

### Annotation Tools

Annotation tools are software applications used to draw bounding boxes or masks and save labels in a structured format.

Examples of open-source annotation tools:

- LabelImg – used for drawing bounding boxes  
- CVAT (Computer Vision Annotation Tool) – supports bounding boxes and segmentation  
- MakeSense.ai – web-based open-source annotation tool  

These tools export annotations in formats compatible with YOLO and other detection models.

Labeling and annotation are essential steps before training object detection and instance segmentation models.


## YOLO (You Only Look Once)

YOLO is a one-stage object detection model that performs object detection in a single forward pass of the network.

Unlike two-stage detectors, YOLO directly predicts:
- Bounding box coordinates
- Object class
- Confidence score

YOLO is fast, efficient, and widely used for real-time object detection and instance segmentation.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="yolo_dataset/dataset.yaml",
    epochs=25,
    imgsz=640,
    batch=8,
    workers=2
)


In [ ]:
# save as validate_seg_labels.py
from pathlib import Path

LABEL_DIRS = [Path("yolo_dataset/labels/train"), Path("yolo_dataset/labels/val")]
ALLOWED_CLASSES = {0}  # change if you have more classes

def is_float(x):
    try:
        float(x)
        return True
    except:
        return False

bad = []
stats = {"files": 0, "rows": 0, "detect_like": 0}

for label_dir in LABEL_DIRS:
    if not label_dir.exists():
        bad.append((str(label_dir), "missing label directory"))
        continue

    for f in sorted(label_dir.glob("*.txt")):
        stats["files"] += 1
        text = f.read_text().strip()

        if not text:
            bad.append((str(f), "empty file"))
            continue

        for ln, line in enumerate(text.splitlines(), 1):
            stats["rows"] += 1
            parts = line.strip().split()

            if not parts:
                bad.append((str(f), f"line {ln}: blank line"))
                continue

            if not all(is_float(p) for p in parts):
                bad.append((str(f), f"line {ln}: non-numeric value"))
                continue

            # class id
            cls = float(parts[0])
            if int(cls) != cls:
                bad.append((str(f), f"line {ln}: class id is not integer ({parts[0]})"))
                continue
            cls = int(cls)
            if cls not in ALLOWED_CLASSES:
                bad.append((str(f), f"line {ln}: unexpected class id {cls}"))
                continue

            n = len(parts)

            # detect format is exactly 5 tokens: class xc yc w h
            if n == 5:
                stats["detect_like"] += 1
                bad.append((str(f), f"line {ln}: looks like DETECT label (5 tokens), not SEGMENT"))
                continue

            if n < 7:
                bad.append((str(f), f"line {ln}: too few tokens for polygon ({n})"))
                continue

            coords = [float(x) for x in parts[1:]]
            if len(coords) % 2 != 0:
                bad.append((str(f), f"line {ln}: odd number of polygon coords ({len(coords)})"))
                continue

            if len(coords) < 6:
                bad.append((str(f), f"line {ln}: fewer than 3 polygon points"))
                continue

            for i, c in enumerate(coords, 1):
                if c < 0 or c > 1:
                    bad.append((str(f), f"line {ln}: coord {i} is out of bounds (0, 1): {c}"))
                    continue


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

model.train(
    data="yolo_dataset/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=8

)

In [ ]:
from ultralytics import YOLO

detect_model = YOLO("models/yolov8n-toilet-detect.pt")

# inference
detect_model.predict(
    source="yolo_dataset/images/val",
    save=True,
    conf=0.25,
    project="runs",
    name="predict_detect"
)

# metrics
detect_model.val(
    data="yolo_dataset/dataset.yaml",
    project="runs",
    name="val_detect"
)

In [ ]:
from ultralytics import YOLO

seg_model = YOLO("models/yolov8n-toilet-seg.pt")

# inference
seg_model.predict(
    source="yolo_dataset/images/val",
    save=True,
    conf=0.25,
    project="runs",
    name="predict_seg"
)

# metrics
seg_model.val(
    data="yolo_dataset/dataset.yaml",
    project="runs",
    name="val_seg"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("models/yolov8n-toilet-seg.pt")

model.val(data="yolo_dataset/dataset.yaml")

## Error Analysis – Misclassified / Imperfect Predictions

No major misclassifications were observed on the validation set.
The model consistently detects and segments toilets correctly.

Minor imperfections were observed in some cases:
- Slightly loose bounding boxes
- Small segmentation mask spillover
- Angle and lighting variations

Overall performance remains strong.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

img_dir = Path("yolo_dataset/images/val")

# Verified lowest-IoU examples from your val set
worst_det = [
    "Primera_b837e709-1987-4df1-a77d-63a351d62baa_e7431bae.jpg",
    "0035_d0530a1f.png",
]
worst_seg = [
    "Primera_b837e709-1987-4df1-a77d-63a351d62baa_e7431bae.jpg",
    "Primera_20a0c01f-5c60-47e1-89c7-b8ae3ed66ee7_beb1a7b0.jpg",
]

det_model = YOLO("models/yolov8n-toilet-detect.pt")
seg_model = YOLO("models/yolov8n-toilet-seg.pt")

rows = []
for task, model, names in [
    ("Detection", det_model, worst_det),
    ("Segmentation", seg_model, worst_seg),
]:
    paths = [str(img_dir / n) for n in names]
    results = model.predict(source=paths, save=False, verbose=False)  # inference only
    for r in results:
        rows.append((task, Path(r.path).name, Image.open(r.path).convert("RGB"), Image.fromarray(r.plot()[..., ::-1])))

fig, axes = plt.subplots(len(rows), 2, figsize=(12, 5 * len(rows)), squeeze=False)

for i, (task, name, orig, pred) in enumerate(rows):
    axes[i, 0].imshow(orig)
    axes[i, 0].set_title(f"Original ({task}): {name}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(pred)
    axes[i, 1].set_title(f"Prediction ({task})")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


Visual inspection of validation images shows that most predictions are correct.
No major misclassifications were observed.
Minor imperfections include slightly loose bounding boxes and small mask spillover.

## Detection – Evaluation Results

This section shows quantitative evaluation of the YOLOv8 detection model
using the validation dataset.

Displayed outputs:
- Confusion Matrix
- Training / validation curves
- Precision, Recall, and mAP trends

These results are generated using YOLO's built-in validation pipeline.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

cm = Image.open("runs/val_detect/confusion_matrix.png")
curve = Image.open("runs/val_detect/BoxF1_curve.png")

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(cm)
plt.title("Detection Confusion Matrix")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(curve)
plt.title("Detection F1 Curve")
plt.axis("off")

plt.show()

## Segmentation – Evaluation Results

This section shows quantitative evaluation of the YOLOv8 instance segmentation model
using the validation dataset.

Displayed outputs:
- Confusion Matrix
- Training / validation curves
- Precision, Recall, and mAP trends

These results are generated using YOLO's built-in validation pipeline.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

cm = Image.open("runs/val_seg/confusion_matrix.png")
mask_curve = Image.open("runs/val_seg/MaskF1_curve.png")

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(cm)
plt.title("Segmentation Confusion Matrix")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask_curve)
plt.title("Segmentation Mask F1 Curve")
plt.axis("off")

plt.show()

## Detection – Precision, Recall and F1 Curves

These plots visualize detection performance across confidence thresholds.
They help analyze trade-offs between precision and recall.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

p_curve = Image.open("runs/val_detect/BoxP_curve.png")
r_curve = Image.open("runs/val_detect/BoxR_curve.png")
f1_curve = Image.open("runs/val_detect/BoxF1_curve.png")

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(p_curve)
plt.title("Detection Precision Curve")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(r_curve)
plt.title("Detection Recall Curve")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(f1_curve)
plt.title("Detection F1 Curve")
plt.axis("off")

plt.show()

## Segmentation – Precision, Recall and F1 Curves

These plots visualize segmentation performance across confidence thresholds.
They help analyze mask quality and localization accuracy.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

p_curve = Image.open("runs/val_seg/MaskP_curve.png")
r_curve = Image.open("runs/val_seg/MaskR_curve.png")
f1_curve = Image.open("runs/val_seg/MaskF1_curve.png")

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(p_curve)
plt.title("Segmentation Precision Curve")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(r_curve)
plt.title("Segmentation Recall Curve")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(f1_curve)
plt.title("Segmentation F1 Curve")
plt.axis("off")

plt.show()

## Detection – Prediction Examples

Sample detection results on validation images showing bounding box localization of toilets.
These examples are used for qualitative inspection of model performance.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

files = os.listdir("runs/predict_detect")[:3]

fig, axs = plt.subplots(1,3, figsize=(12,4))

for i,f in enumerate(files):
    img = Image.open(f"runs/predict_detect/{f}")
    axs[i].imshow(img)
    axs[i].axis("off")

plt.show()

## Segmentation – Prediction Examples

Sample instance segmentation results on validation images showing predicted toilet masks.
These examples are used for qualitative inspection of mask quality and localization.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

files = os.listdir("runs/predict_seg")[:3]

fig, axs = plt.subplots(1,3, figsize=(12,4))

for i,f in enumerate(files):
    img = Image.open(f"runs/predict_seg/{f}")
    axs[i].imshow(img)
    axs[i].axis("off")

plt.show()

## Saved Models

Best-performing models were saved for inference:

- models/yolov8n-toilet-detect.pt  
- models/yolov8n-toilet-seg.pt